In [ ]:
import torch
from networks import EMCADNet

print("MPS available:", torch.backends.mps.is_available())
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


True


In [28]:
!mkdir -p ./data/synapse

!unzip -q synapse.zip -d ./data/synapse/

unzip:  cannot find or open synapse.zip, synapse.zip.zip or synapse.zip.ZIP.


In [20]:
model = torch.load("/Users/sumankar/Downloads/emcad_model.pth", map_location=device)
print(model)


OrderedDict([('encoder.stage1.0.weight', tensor([[[[-0.2449, -0.2325, -0.0998],
          [ 0.0486,  0.1637,  0.2506],
          [-0.2336, -0.2311,  0.2168]]],


        [[[-0.2951,  0.1063,  0.2380],
          [-0.0788,  0.3035,  0.1155],
          [ 0.1417,  0.0426,  0.0365]]],


        [[[ 0.0777,  0.1098, -0.2302],
          [ 0.3173,  0.1564,  0.1865],
          [-0.1884, -0.0632, -0.0291]]],


        [[[-0.1221,  0.1522, -0.2998],
          [ 0.1693,  0.0290, -0.1340],
          [ 0.3059, -0.1717, -0.2296]]],


        [[[ 0.0387,  0.1843, -0.2617],
          [-0.2092, -0.2961,  0.0460],
          [-0.3471, -0.2293,  0.1486]]],


        [[[ 0.2378,  0.3418, -0.0192],
          [-0.1565,  0.3110,  0.2237],
          [-0.1431, -0.2275,  0.1539]]],


        [[[-0.3026, -0.3330,  0.1792],
          [-0.1603, -0.2381, -0.0420],
          [-0.2512,  0.1562, -0.1138]]],


        [[[ 0.0710, -0.2947,  0.0684],
          [ 0.2599, -0.2865, -0.0139],
          [-0.0141, -0.2918, -0.12

In [21]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap

print("PyTorch:", torch.__version__)

print("MPS Available:",
      torch.backends.mps.is_available())

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using Device:", device)

PyTorch: 2.8.0
MPS Available: True
Using Device: mps


In [ ]:
import os
import glob
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader


# =========================================================
# DATASET
# =========================================================

class SynapseDataset(Dataset):

    def __init__(self, data_dir):

        self.files = glob.glob(data_dir + "/*.npz")

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        data = np.load(self.files[idx])

        image = data["image"]
        mask = data["label"]

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        # [H,W] -> [1,H,W]
        image = image.unsqueeze(0)

        return image, mask


data_dir = "./data/synapse/synapse/train_npz_new"
train_dataset = SynapseDataset(data_dir)
print(f"Found {len(train_dataset)} .npz files in {data_dir}")
if len(train_dataset) == 0:
    raise FileNotFoundError(
        f"No .npz files found in {data_dir}.\n"
        "Put your Synapse .npz files there or update the path."
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# =========================================================
# CAB
# =========================================================

class CAB(nn.Module):

    def __init__(self, channels, reduction=16):

        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))

        attention = avg_out + max_out
        attention = self.sigmoid(attention)

        return x * attention

# =========================================================
# SAB
# =========================================================

class SAB(nn.Module):

    def __init__(self, kernel_size=7):

        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)

        attn = torch.cat([avg, mx], dim=1)

        attn = self.sigmoid(self.conv(attn))

        return x * attn

# =========================================================
# MSDC
# =========================================================

class MSDC(nn.Module):

    def __init__(self, channels, kernels=(1,3,5), stride=1):

        super().__init__()

        self.branches = nn.ModuleList()

        for k in kernels:

            self.branches.append(
                nn.Sequential(
                    nn.Conv2d(
                        channels,
                        channels,
                        kernel_size=k,
                        stride=stride,
                        padding=k//2,
                        groups=channels,
                        bias=False
                    ),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(inplace=True)
                )
            )

    def forward(self, x):

        outputs = []

        for branch in self.branches:

            outputs.append(branch(x))

        return outputs

# =========================================================
# MSCB
# =========================================================

class MSCB(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        kernels=(1,3,5),
        expansion=2
    ):

        super().__init__()

        self.use_skip = (
            stride == 1 and
            in_channels == out_channels
        )

        hidden = in_channels * expansion

        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True)
        )

        self.msdc = MSDC(hidden, kernels, stride)

        self.project = nn.Sequential(
            nn.Conv2d(
                hidden * len(kernels),
                out_channels,
                1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):

        identity = x

        x = self.expand(x)

        x = torch.cat(self.msdc(x), dim=1)

        x = self.project(x)

        if self.use_skip:

            x = x + identity

        return x

# =========================================================
# EUCB
# =========================================================

class EUCB(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.up = nn.Upsample(
            scale_factor=2,
            mode='bilinear',
            align_corners=False
        )

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                3,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels,
                out_channels,
                1,
                bias=False
            )
        )

    def forward(self, x):

        x = self.up(x)

        x = self.block(x)

        return x

# =========================================================
# LGAG
# =========================================================

class LGAG(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):

        g1 = self.W_g(g)
        x1 = self.W_x(x)

        psi = self.relu(g1 + x1)

        psi = self.psi(psi)

        return x * psi

# =========================================================
# ENCODER
# =========================================================

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.stage1 = nn.Sequential(
            nn.Conv2d(1, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.stage2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.stage3 = nn.Sequential(
            nn.Conv2d(128, 320, 3, stride=2, padding=1),
            nn.BatchNorm2d(320),
            nn.ReLU(inplace=True)
        )

        self.stage4 = nn.Sequential(
            nn.Conv2d(320, 512, 3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):

        x1 = self.stage1(x)
        x2 = self.stage2(x1)
        x3 = self.stage3(x2)
        x4 = self.stage4(x3)

        return x4, [x3, x2, x1]

# =========================================================
# EMCAD DECODER
# =========================================================

class EMCAD(nn.Module):

    def __init__(self):

        super().__init__()

        self.cab = CAB(512)
        self.sab = SAB()

        self.u3 = EUCB(512, 320)
        self.g3 = LGAG(320, 320, 160)
        self.m3 = MSCB(320, 320)

        self.u2 = EUCB(320, 128)
        self.g2 = LGAG(128, 128, 64)
        self.m2 = MSCB(128, 128)

        self.u1 = EUCB(128, 64)
        self.g1 = LGAG(64, 64, 32)
        self.m1 = MSCB(64, 64)

    def forward(self, x, skips):

        x = self.cab(x)
        x = self.sab(x)

        d4 = x

        d3 = self.u3(d4)
        d3 = self.g3(d3, skips[0])
        d3 = self.m3(d3)

        d2 = self.u2(d3)
        d2 = self.g2(d2, skips[1])
        d2 = self.m2(d2)

        d1 = self.u1(d2)
        d1 = self.g1(d1, skips[2])
        d1 = self.m1(d1)

        return d4, d3, d2, d1

# =========================================================
# FULL MODEL
# =========================================================

class EMCAD_Model(nn.Module):

    def __init__(self, num_classes=14):

        super().__init__()

        self.encoder = Encoder()

        self.decoder = EMCAD()

        self.head_d4 = nn.Conv2d(512, num_classes, 1)
        self.head_d3 = nn.Conv2d(320, num_classes, 1)
        self.head_d2 = nn.Conv2d(128, num_classes, 1)
        self.head_d1 = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):

        x, skips = self.encoder(x)

        d4, d3, d2, d1 = self.decoder(x, skips)

        p4 = self.head_d4(d4)
        p3 = self.head_d3(d3)
        p2 = self.head_d2(d2)
        p1 = self.head_d1(d1)

        return [p4, p3, p2, p1]

# =========================================================
# MODEL

# =========================================================


model = EMCAD_Model(num_classes=14).to(device)

ValueError: num_samples should be a positive integer value, but got num_samples=0